<a href="https://colab.research.google.com/github/dev09saransh/dev09saransh/blob/main/bitcoin_analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
import pandas as pd
import numpy as np
import os

# ----------------------------
# File paths
# ----------------------------
sent_path = "/fear_greed_index.csv"
trades_path = "/historical_data.csv"

print("Loading files:", sent_path, trades_path)

# ----------------------------
# Load data
# ----------------------------
sent = pd.read_csv(sent_path, low_memory=False)
trades = pd.read_csv(trades_path, low_memory=False)

print("Sentiment shape:", sent.shape)
print("Trades shape:", trades.shape)
print("Sentiment columns:", sent.columns.tolist())
print("Trades columns:", trades.columns.tolist())

# ----------------------------
# Identify relevant columns
# ----------------------------
sent_date_col = "date"
sent_class_col = "classification"
trade_time_col = "Timestamp IST"
account_col = "Account"
closed_pnl_col = "Closed PnL" if "Closed PnL" in trades.columns else None
price_col = "Execution Price"
side_col = "Side"

print("Detected columns:\n sentiment date:", sent_date_col,
      "sent class:", sent_class_col,
      "\n trade time:", trade_time_col,
      "account:", account_col,
      "closedPnL:", closed_pnl_col,
      "\n size:", None,
      "price:", price_col,
      "leverage:", None,
      "side:", side_col)

print("Sent date nulls:", sent[sent_date_col].isna().sum(), "of", len(sent))
print("Trade time nulls:", trades[trade_time_col].isna().sum(), "of", len(trades))

# ----------------------------
# Convert and clean sentiment data
# ----------------------------
sent[sent_date_col] = pd.to_datetime(sent[sent_date_col], errors="coerce").dt.date

# Map sentiment to numeric
sentiment_map = {
    "Extreme Fear": 0,
    "Fear": 25,
    "Neutral": 50,
    "Greed": 75,
    "Extreme Greed": 100
}
sent["sentiment_num"] = sent[sent_class_col].map(sentiment_map)

print("Unique sentiment classification values (sample):")
print(sent[sent_class_col].dropna().unique())

# ----------------------------
# Convert and clean trades data
# ----------------------------
# Convert Timestamp IST to datetime (keeping only date for merging)
trades["trade_datetime"] = pd.to_datetime(trades[trade_time_col], errors="coerce")
trades["trade_date"] = trades["trade_datetime"].dt.date

# Optional: Drop timezone confusion (IST → date only, so no tz convert here)
trades["side_str"] = trades[side_col].astype(str).str.lower()

# ----------------------------
# Merge trades with sentiment
# ----------------------------
merged = trades.merge(
    sent[[sent_date_col, sent_class_col, "sentiment_num"]],
    left_on="trade_date",
    right_on=sent_date_col,
    how="left"
)

print("Merged shape:", merged.shape)

# ----------------------------
# Simple per-sentiment summary
# ----------------------------
if closed_pnl_col and closed_pnl_col in merged.columns:
    summary = merged.groupby(sent_class_col).agg(
        trades_count=(closed_pnl_col, "count"),
        mean_pnl=(closed_pnl_col, "mean"),
        winrate=(closed_pnl_col, lambda x: np.mean(x > 0)),
    ).reset_index()
else:
    summary = pd.DataFrame()

print("\nPer-sentiment summary:\n", summary)

# ----------------------------
# Save sample for verification
# ----------------------------
sample_csv = "merged_trades_sentiment_sample.csv"
merged.head(100).to_csv(sample_csv, index=False)
print(f"Saved merged sample CSV to: {sample_csv}")



Loading files: /fear_greed_index.csv /historical_data.csv
Sentiment shape: (2644, 4)
Trades shape: (211224, 16)
Sentiment columns: ['timestamp', 'value', 'classification', 'date']
Trades columns: ['Account', 'Coin', 'Execution Price', 'Size Tokens', 'Size USD', 'Side', 'Timestamp IST', 'Start Position', 'Direction', 'Closed PnL', 'Transaction Hash', 'Order ID', 'Crossed', 'Fee', 'Trade ID', 'Timestamp']
Detected columns:
 sentiment date: date sent class: classification 
 trade time: Timestamp IST account: Account closedPnL: Closed PnL 
 size: None price: Execution Price leverage: None side: Side
Sent date nulls: 0 of 2644
Trade time nulls: 0 of 211224
Unique sentiment classification values (sample):
['Fear' 'Extreme Fear' 'Neutral' 'Greed' 'Extreme Greed']
Merged shape: (211224, 22)

Per-sentiment summary:
   classification  trades_count    mean_pnl   winrate
0   Extreme Fear          2326    1.891632  0.292777
1  Extreme Greed          5621  205.816345  0.553282
2           Fear      